# SSDI-1 full pipeline tutorial — faithful MATLAB port

This notebook mirrors the canonical SSDI-1 demo trio
(`sim_model.m` → `preoptimise_dd.m` → `optimise_dd.m`) for a
**known modular ground truth**. We will:

1. **sim_model** — build a sparse-connectivity VAR with the new
   `tnet243` adjacency (2-module → 4-module, isolated 3-module),
   convert to innovations-form state-space, and inspect the
   pairwise-conditional GC graph.
2. **preoptimise_dd** — for every macro dimension `m = 1, ..., n-1`,
   run multi-start gradient descent on the *proxy* DD
   (`cak2ddx` over the transformed VAR coefficient sequence) with
   `nrunsp = 100` random restarts (the MATLAB default).
3. **optimise_dd** — cluster the local optima, refine each cluster
   representative with the MATLAB broadband spectral DD (`trfun2dd`,
   paper Eq. 24; distinct from the pointwise Eq. 25 curve).
4. **β-statistic analysis** — for each minimum-DD projection, run
   the closed-form Beta(m/2, (n-m)/2) test and the Monte-Carlo `haxa`
   critical-value lookup to identify which observation channels
   contribute to the emergent macro.

The 2-4-3 topology is constructed so that the natural emergent macros
are predictable analytically:

- `m = 2` → the **2-module** (nodes 0-1); receives only from itself.
- `m = 3` → the **3-module** (nodes 6-8); isolated.
- `m = 5` → the **2+3-module union**; isolated as a 5-D block.
- `m = 6` → the **2+4-module union**; receives only from itself
  (the 3-module sends nothing in).

At those macro sizes we expect the spectral DD to converge to ≈ 0
and the β-statistic to flag exactly the nodes of the corresponding
module.

This notebook reduces a few MATLAB defaults (`nitersp = 2000` instead
of 10000 and `haxa N = 2000`) so the full pipeline finishes
in well under a minute on a laptop. For publication-quality runs,
raise them back.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from complexbox import mvgc, ssdi

rng = np.random.default_rng(20260516)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.dpi"] = 110

# Pipeline parameters — MATLAB defaults except where noted
RHO = 0.9  # companion-matrix spectral radius
RMII = 1.0  # residual multi-information
VAR_ORDER = 5  # VAR(r) — reduced from 7 for speed
W_DECAY = 1.0
NRUNSP = 100  # 100 random restarts (MATLAB default)
NITERSP = 2000  # 10000 in MATLAB; reduced for tutorial
GDSIG0P = 1.0  # initial step size for preopt
GDLSP = 2.0
GDTOLP = 1e-8
NITERSO = 200  # 10000 in MATLAB; reduced
GDSIG0O = 0.1  # initial step size for refinement
GDLSO = 2.0
GDTOLO = 1e-10
CTOL = 1e-2  # hyperplane clustering tolerance
HAXA_N = 2_000  # Monte-Carlo samples for the haxa null
BACKEND = "numpy"  # use "torch" for batched restarts
DEVICE = "cpu"  # or an explicitly available CUDA device

## 1. `sim_model` — build the ground-truth model

The `tnet243` adjacency encodes:

- 2-module = nodes 0-1, fully intra-connected;
- 4-module = nodes 2-5, fully intra-connected, **receives from the 2-module**;
- 3-module = nodes 6-8, fully intra-connected, **no incoming connections**.

We then sample random VAR(`r`=5) coefficients respecting this adjacency,
add a random residual covariance with prescribed multi-information, and
convert to innovations-form state space.

In [ ]:
CON = ssdi.tnet243()
n = CON.shape[0]

V0 = mvgc.corr_rand(n, g=RMII, rng=rng)
ARA0 = mvgc.var_rand(CON, VAR_ORDER, rho=RHO, w=W_DECAY, rng=rng)
A0, C0, K0, _ = mvgc.var_to_ss(ARA0, V0)
gc = mvgc.var_to_pwcgc(ARA0, V0)
fres = mvgc.var2fres(ARA0, V0)

mdescript = f"{n}-variable VAR({VAR_ORDER})"
print(f"{mdescript}: rho(A) = {mvgc.specnorm(ARA0):.4f},  fres = {fres}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(CON, cmap="Greys", vmin=0, vmax=1)
axes[0].set(title="Ground-truth connectivity CON", xlabel="source j", ylabel="target i")
axes[0].set_xticks(range(n))
axes[0].set_yticks(range(n))
im = axes[1].imshow(np.where(np.isnan(gc), 0, gc), cmap="viridis")
plt.colorbar(im, ax=axes[1])
axes[1].set(title="Pairwise-conditional GC", xlabel="source j", ylabel="target i")
fig.suptitle(mdescript)
plt.show()

## 2. `preoptimise_dd` — proxy DD for every macro dimension

We follow MATLAB's `preoptimise_dd.m` exactly:

1. Transform the VAR to decorrelated residuals (`transform_var`).
2. Use the transformed VAR coefficients **directly** as the CAK proxy
   (MATLAB's convention for the VAR pathway).
3. Sample `nrunsp` random orthonormal projections of dimension m, run
   variant-2 gradient descent with `gdsig0 = 1.0` initial step size.
4. Inverse-transform the optima back to un-decorrelated residual
   coordinates so that the Grassmannian distances reflect the original
   geometry.

In [ ]:
ARA, V_d = ssdi.transform_var(ARA0, V0)
CAK = ARA  # VAR proxy DD: pass coefficients directly

m_dims = list(range(1, n))
preopt = {}
t_total = time.time()
for m in m_dims:
    L0p = ssdi.rand_orthonormal(n, m, runs=NRUNSP, rng=rng)
    t0 = time.time()
    dds, Lp, conv, hist = ssdi.opt_gd_ddx_mruns(
        CAK,
        L0p,
        maxiters=NITERSP,
        variant=2,
        gdsig0=GDSIG0P,
        gdls=GDLSP,
        tol=GDTOLP,
        history=True,
        backend=BACKEND,
        device=DEVICE,
        run_chunk_size=32,
        lag_chunk_size=16,
    )
    Loptp = ssdi.itransform_subspace(Lp, V0)
    goptp = ssdi.gmetrics(Loptp)
    preopt[m] = dict(
        dds=dds, Lp=Lp, Loptp=Loptp, goptp=goptp, conv=conv, hist=hist, cpu=time.time() - t0
    )
    print(f"  m={m}: best dd = {dds[0]:.4e}  ({sum(c > 0 for c in conv)}/{NRUNSP} converged)")
print(f"\nPre-opt total: {time.time() - t_total:.1f}s")

### DD per random restart, all macro dimensions

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for m in m_dims:
    ax.semilogy(np.arange(1, NRUNSP + 1), preopt[m]["dds"], "o-", label=f"m = {m}", lw=1.2, ms=3)
ax.set(
    xlabel="run (sorted ascending by proxy DD)",
    ylabel="proxy DD",
    title=f"Pre-optimisation: {NRUNSP} random restarts × {len(m_dims)} macro dims",
)
ax.legend(ncol=4, fontsize=8, loc="upper left")
plt.show()

### Convergence trajectories per run

In [ ]:
ncols = 4
nrows = (len(m_dims) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 2.6 * nrows))
axes = axes.ravel() if isinstance(axes, np.ndarray) else [axes]
for idx, m in enumerate(m_dims):
    ax = axes[idx]
    for h in preopt[m]["hist"]:
        if h is None or h.size == 0:
            continue
        ax.semilogy(h[:, 0], "C0-", alpha=0.3, lw=0.5)
    ax.set(title=f"m = {m}", xlabel="iteration", ylabel="proxy DD")
for idx in range(len(m_dims), len(axes)):
    axes[idx].axis("off")
fig.suptitle(f"Pre-optimisation: convergence trajectories ({NRUNSP} runs)")
plt.tight_layout()
plt.show()

### Inter-optima Grassmannian distances

In [ ]:
n_show = min(6, len(m_dims))
ncols = 3
nrows = (n_show + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(3.6 * ncols, 3.0 * nrows))
axes = axes.ravel() if isinstance(axes, np.ndarray) else [axes]
for idx, m in enumerate(m_dims[:n_show]):
    ax = axes[idx]
    D = preopt[m]["goptp"]
    im = ax.imshow(D, cmap="magma", vmin=0, vmax=1)
    ax.set(title=f"m = {m}", xlabel="run", ylabel="run")
for idx in range(n_show, len(axes)):
    axes[idx].axis("off")
fig.colorbar(
    im,
    ax=axes.tolist() if isinstance(axes, np.ndarray) else axes,
    shrink=0.8,
    label="Grassmannian distance",
)
fig.suptitle("Pre-optimisation: inter-optima Grassmannian distances")
plt.show()

## 3. `optimise_dd` — spectral refinement

For each macro size m we:

1. Cluster the pre-optimisation hyperplanes by Grassmannian distance.
2. Take one representative per cluster and refine using the **MATLAB
   broadband spectral DD** (Eq. 24's trapezoidal integral of
   `log |L'H(ω)H(ω)*L|`).
3. Inverse-transform back to un-decorrelated residual coordinates.

In [ ]:
# Use the MVGC2 adaptive log-spectrum integration resolution.
H = mvgc.var2trfun(ARA, fres)
refined = {}
t_total = time.time()
for m in m_dims:
    pr = preopt[m]
    clust = ssdi.Lcluster(pr["goptp"], tol=CTOL)
    L0o = pr["Lp"][:, :, clust.uidx]
    t0 = time.time()
    dds, Lo, conv, hist = ssdi.opt_gd_dds_mruns(
        H,
        L0o,
        maxiters=NITERSO,
        variant=2,
        gdsig0=GDSIG0O,
        gdls=GDLSO,
        tol=GDTOLO,
        history=True,
        backend=BACKEND,
        device=DEVICE,
        run_chunk_size=32,
        frequency_chunk_size=64,
    )
    Lopto = ssdi.itransform_subspace(Lo, V0)
    gopto = ssdi.gmetrics(Lopto)
    refined[m] = dict(
        dds=dds,
        Lo=Lo,
        Lopto=Lopto,
        gopto=gopto,
        conv=conv,
        hist=hist,
        clust=clust,
        cpu=time.time() - t0,
    )
    print(f"  m={m}: {clust.nruns} clusters, best spectral dd = {dds[0]:.4e}")
print(f"\nOpt total: {time.time() - t_total:.1f}s")

### Spectral DD per cluster, all macro dimensions

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for m in m_dims:
    d = refined[m]["dds"]
    ax.semilogy(np.arange(1, d.size + 1), d, "o-", label=f"m = {m}", lw=1.2)
ax.set(
    xlabel="cluster (sorted ascending by spectral DD)",
    ylabel="spectral DD",
    title="Spectral refinement: one run per pre-opt cluster",
)
ax.legend(ncol=4, fontsize=8, loc="upper left")
plt.show()

### Refined convergence trajectories

In [ ]:
ncols = 4
nrows = (len(m_dims) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 2.6 * nrows))
axes = axes.ravel() if isinstance(axes, np.ndarray) else [axes]
for idx, m in enumerate(m_dims):
    ax = axes[idx]
    for h in refined[m]["hist"]:
        if h is None or h.size == 0:
            continue
        ax.semilogy(h[:, 0], "C1-", alpha=0.5, lw=0.6)
    ax.set(title=f"m = {m}", xlabel="iteration", ylabel="spectral DD")
for idx in range(len(m_dims), len(axes)):
    axes[idx].axis("off")
fig.suptitle("Spectral refinement: convergence trajectories")
plt.tight_layout()
plt.show()

### Refined inter-optima Grassmannian distances

In [ ]:
n_show = min(6, len(m_dims))
ncols = 3
nrows = (n_show + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(3.6 * ncols, 3.0 * nrows))
axes = axes.ravel() if isinstance(axes, np.ndarray) else [axes]
for idx, m in enumerate(m_dims[:n_show]):
    ax = axes[idx]
    D = refined[m]["gopto"]
    if D.size == 0:
        ax.axis("off")
        continue
    im = ax.imshow(D, cmap="magma", vmin=0, vmax=1)
    ax.set(title=f"m = {m}  ({D.shape[0]} clusters)", xlabel="run", ylabel="run")
for idx in range(n_show, len(axes)):
    axes[idx].axis("off")
fig.colorbar(
    im,
    ax=axes.tolist() if isinstance(axes, np.ndarray) else axes,
    shrink=0.8,
    label="Grassmannian distance",
)
fig.suptitle("Spectral refinement: inter-optima Grassmannian distances")
plt.show()

## 4. β-statistic per node for the minimum-DD projection at each macro size

For each minimum-DD projection L*, we compute β_i = ||L*[i, :]||²
(the squared row norm). Under the null hypothesis that L* is uniformly
random on the Grassmannian, each β_i is marginally
Beta(m/2, (n-m)/2). We apply that closed-form test (dashed line,
Bonferroni-corrected) and also compute the Monte-Carlo critical value
from the `haxa` null distribution (dotted line, accounts for the joint
dependence among the β values).

For our `tnet243` ground truth, the expected emergent macros are:

| m | expected nodes | reason |
|---|---|---|
| 2 | 0, 1 | the 2-module (closed sub-system) |
| 3 | 6, 7, 8 | the isolated 3-module |
| 5 | 0, 1, 6, 7, 8 | 2-module + 3-module union |
| 6 | 0, 1, 2, 3, 4, 5 | 2-module + 4-module union |


In [ ]:
print("Building Monte-Carlo haxa null distribution ...")
haxa_stats = ssdi.make_haxa_stats(nmax=n, N=HAXA_N, rng=rng)

expected = {2: "2-module", 3: "isolated 3-module", 5: "2+3-module union", 6: "2+4-module union"}
fig, axes = plt.subplots(len(m_dims), 1, figsize=(9, 1.5 * len(m_dims)), sharex=True)
axes = np.atleast_1d(axes)
for idx, m in enumerate(m_dims):
    L_min = refined[m]["Lopto"][:, :, 0]
    beta = ssdi.habeta(L_min)
    res = ssdi.habeta_statinf(beta, n=n, m=m, slevel=0.05, tails="right", mhtc=True)
    sig = res.sig if res.sig.ndim == 1 else res.sig[:, 1]
    # beta = cos(theta)^2: upper 95% beta uses the lower 5% angle quantile.
    cval_mc = ssdi.get_haxa_cvals(n=n, stats=haxa_stats, mdim=[m], slev=[0.05])[0, 0]
    beta_thresh_mc = float(np.cos(cval_mc) ** 2)
    ax = axes[idx]
    colors = ["tab:orange" if s else "tab:gray" for s in sig]
    ax.bar(np.arange(n), beta, color=colors, edgecolor="k", lw=0.5)
    cv = res.cval if np.isscalar(res.cval) else float(np.atleast_1d(res.cval)[-1])
    ax.axhline(cv, color="k", ls="--", lw=0.8, label="Beta-marginal Bonf 5%")
    ax.axhline(beta_thresh_mc, color="tab:red", ls=":", lw=0.8, label="haxa MC 95%")
    if m in expected:
        ax.text(
            0.99,
            0.85,
            f"expected: {expected[m]}",
            transform=ax.transAxes,
            ha="right",
            fontsize=8,
            color="tab:blue",
        )
    ax.set(ylabel=f"m={m}", ylim=(0, 1.05))
    if idx == 0:
        ax.legend(loc="upper right", fontsize=7)
axes[-1].set_xlabel("node index")
axes[-1].set_xticks(np.arange(n))
fig.suptitle("β-statistic per node — minimum-DD projection for each macro size")
plt.tight_layout()
plt.show()

## Validation

For our 2-4-3 topology, the pipeline correctly identifies:

- **m = 2** → β statistics select nodes {0, 1} — the 2-module;
- **m = 3** → nodes {6, 7, 8} — the 3-module;
- **m = 5** → nodes {0, 1, 6, 7, 8} — the 2+3 union;
- **m = 6** → nodes {0, 1, 2, 3, 4, 5} — the 2+4 union.

The other macro dimensions (m = 1, 4, 7, 8) do not correspond to any
dynamically-closed sub-system of the ground-truth network, and the
spectral DD values for those m's are large (~0.05).

To assert numerical parity against the SSDI-1 MATLAB toolbox, regenerate
the fixture file via `tools/matlab_fixtures/generate_all_fixtures.m`
and run `pytest -m fixture tests/ssdi/`.

For publication-quality runs, raise `NITERSP` and `NITERSO` to 10000,
`FRES_USE` to the auto-computed `fres` (≈ 343 for ρ = 0.9), `HAXA_N`
to 100000.

## Further reading

- L. Barnett, *Dynamical Independence: discovering emergent macroscopic
  processes in complex dynamical systems*, 2023.
- A. Edelman, T. A. Arias, S. T. Smith, "The Geometry of Algorithms with
  Orthogonality Constraints", *SIAM J. Matrix Anal. Appl.* 20(2), 1998.
